# Translation Task

[video](https://www.youtube.com/watch?v=ISNdQcPhsts)

In [1]:
from pathlib import Path

from dotenv import load_dotenv
from torch.utils.tensorboard import SummaryWriter

import models.deep_learning.architectures.transformer.tasks.translation as trn

load_dotenv()
device = trn.get_device()

## Config

In [2]:
CONFIG = trn.Config(
    batch_size=8,
    num_epochs=50,
    lr=1e-4,
    src_seq_len=350,
    tgt_seq_len=350,
    d_model=512,
    datasource="Helsinki-NLP/opus_books",
    src_lang="en",
    tgt_lang="es",
)


In [3]:
writer = SummaryWriter(CONFIG.experiment_name)
Path(f"{CONFIG.datasource}_{CONFIG.model_folder}").mkdir(parents=True, exist_ok=True)

## Load Dataset (from HuggingFace)

In [4]:
raw_ds = trn.TranslationHFDataset.load_dataset(
    path=CONFIG.datasource,
    name=f"{CONFIG.src_lang}-{CONFIG.tgt_lang}",
    split="train",
)

## Tokenization

In [5]:
tokenizer_src = trn.get_or_build_tokenizer(
    Path(CONFIG.tokenizer_src_file), raw_ds, CONFIG.src_lang
)
tokenizer_tgt = trn.get_or_build_tokenizer(
    Path(CONFIG.tokenizer_tgt_file), raw_ds, CONFIG.tgt_lang
)

In [6]:
original_len = len(raw_ds)

raw_ds = raw_ds.filter(
    lambda x: (
        len(tokenizer_src.encode(x["translation"][CONFIG.src_lang]).ids)
        <= CONFIG.src_seq_len - 2
        and len(tokenizer_tgt.encode(x["translation"][CONFIG.tgt_lang]).ids)
        <= CONFIG.tgt_seq_len - 1
    )
)

filtered_len = len(raw_ds)
removal_percentage = (original_len - filtered_len) / original_len * 100
print(f"Original dataset size: {original_len}")
print(f"Filtered dataset size: {filtered_len}")
print(f"Removed: {original_len - filtered_len} samples ({removal_percentage:.2f}%)")

Filter:   0%|          | 0/93470 [00:00<?, ? examples/s]

Original dataset size: 93470
Filtered dataset size: 93464
Removed: 6 samples (0.01%)


## Create dataloaders

In [7]:
train_dataloader, val_dataloader = trn.create_dataloaders(
    raw_ds, tokenizer_src, tokenizer_tgt, CONFIG
)

## Create model

In [8]:
model = trn.Translator(
    src_vocab_size=tokenizer_src.get_vocab_size(),
    tgt_vocab_size=tokenizer_tgt.get_vocab_size(),
    src_max_length=CONFIG.src_seq_len,
    tgt_max_length=CONFIG.tgt_seq_len,
    embed_size=CONFIG.d_model,
).to(device)

## Train the model

In [9]:
from collections.abc import Callable

import torch
import torch.nn as nn
from tokenizers import Tokenizer
from torch.utils.data import DataLoader
from tqdm import tqdm


In [10]:
import shutil

import torchmetrics.text


def greedy_decode(
    model: trn.Translator,
    source: torch.Tensor,
    source_mask: torch.Tensor,
    tokenizer_tgt: Tokenizer,
    tgt_max_len: int,
    device: torch.device,
):
    sos_idx = tokenizer_tgt.token_to_id("[SOS]")
    eos_idx = tokenizer_tgt.token_to_id("[EOS]")

    encoder_output = model.encode(source, source_mask)
    decoder_input = torch.empty(1, 1).fill_(sos_idx).type_as(source).to(device)
    while True:
        if decoder_input.size(1) == tgt_max_len:
            break

        decoder_mask = trn.causal_mask(decoder_input.size(1)).unsqueeze(0).to(device)

        out = model.decode(decoder_input, encoder_output, decoder_mask, source_mask)
        prob = model.project(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        decoder_input = torch.cat(
            [
                decoder_input,
                torch.empty(1, 1).type_as(source).fill_(next_word.item()).to(device),
            ],
            dim=1,
        )

        if next_word == eos_idx:
            break

    return decoder_input.squeeze(0)


def run_validation(
    model: trn.Translator,
    validation_ds: DataLoader[dict[str, torch.Tensor | str]],
    tokenizer_src: Tokenizer,
    tokenizer_tgt: Tokenizer,
    src_max_len: int,
    tgt_max_len: int,
    device: torch.device,
    print_msg: Callable[[str], None],
    global_step: int,
    writer: SummaryWriter | None,
    num_examples: int = 2,
):
    model.eval()
    count = 0
    source_texts: list[str] = []
    expected: list[str] = []
    predicted: list[str] = []

    console_width = shutil.get_terminal_size().columns

    with torch.no_grad():
        for batch in validation_ds:
            count += 1
            encoder_input = batch["encoder_input"].to(device)
            encoder_mask = batch["encoder_mask"].to(device)
            assert encoder_input.size(0) == 1, "Batch size must be 1 for validation"

            model_out = greedy_decode(
                model, encoder_input, encoder_mask, tokenizer_tgt, tgt_max_len, device
            )

            source_text = batch["src_text"][0]
            target_text = batch["tgt_text"][0]
            model_out_text = tokenizer_tgt.decode(model_out.detach().cpu().numpy())

            source_texts.append(source_text)
            expected.append(target_text)
            predicted.append(model_out_text)

            print_msg("-" * console_width)
            print_msg(f"{'SOURCE: ':>12}{source_text}")
            print_msg(f"{'TARGET: ':>12}{target_text}")
            print_msg(f"{'PREDICTED: ':>12}{model_out_text}")

            if count == num_examples:
                print_msg("-" * console_width)
                break

    if writer:
        cer = torchmetrics.text.CharErrorRate()(predicted, expected)
        writer.add_scalar("validation cer", cer, global_step)
        wer = torchmetrics.text.WordErrorRate()(predicted, expected)
        writer.add_scalar("validation wer", wer, global_step)
        bleu = torchmetrics.text.BLEUScore()(predicted, expected)
        writer.add_scalar("validation BLEU", bleu, global_step)
        writer.flush()


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG.lr, eps=1e-9)

# If the user specified a model to preload before training, load it
initial_epoch = 0
global_step = 0
preload = CONFIG.preload
model_filename = (
    trn.latest_weights_file_path(CONFIG)
    if preload == "latest"
    else trn.get_weights_file_path(CONFIG, preload)
    if preload
    else None
)
if model_filename:
    print(f"Preloading model {model_filename}")
    state = torch.load(model_filename)
    model.load_state_dict(state["model_state_dict"])
    initial_epoch = state["epoch"] + 1
    optimizer.load_state_dict(state["optimizer_state_dict"])
    global_step = state["global_step"]
else:
    print("No model to preload, starting from scratch")

loss_fn = nn.CrossEntropyLoss(
    ignore_index=tokenizer_src.token_to_id("[PAD]"), label_smoothing=0.1
).to(device)

for epoch in range(initial_epoch, CONFIG.num_epochs):
    torch.cuda.empty_cache()
    model.train()
    batch_iterator = tqdm(train_dataloader, desc=f"Processing Epoch {epoch:02d}")
    for i, batch in enumerate(batch_iterator):
        if i > 5:
            break
        encoder_input = batch["encoder_input"].to(device)  # (b, seq_len)
        decoder_input = batch["decoder_input"].to(device)  # (B, seq_len)
        encoder_mask = batch["encoder_mask"].to(device)  # (B, 1, 1, seq_len)
        decoder_mask = batch["decoder_mask"].to(device)  # (B, 1, seq_len, seq_len)

        # Run the tensors through the encoder, decoder and the projection layer
        encoder_output = model.encode(
            encoder_input, encoder_mask
        )  # (B, seq_len, d_model)
        decoder_output = model.decode(
            decoder_input, encoder_output, decoder_mask, encoder_mask
        )  # (B, seq_len, d_model)
        proj_output = model.project(decoder_output)  # (B, seq_len, vocab_size)

        # Compare the output with the label
        label = batch["label"].to(device)  # (B, seq_len)

        # Compute the loss using a simple cross entropy
        loss = loss_fn(
            proj_output.view(-1, tokenizer_tgt.get_vocab_size()), label.view(-1)
        )
        batch_iterator.set_postfix({"loss": f"{loss.item():6.3f}"})

        # Log the loss
        writer.add_scalar("train loss", loss.item(), global_step)
        writer.flush()

        # Backpropagate the loss
        loss.backward()

        # Update the weights
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        global_step += 1

    # Run validation at the end of every epoch
    run_validation(
        model,
        val_dataloader,
        tokenizer_src,
        tokenizer_tgt,
        CONFIG.src_seq_len,
        CONFIG.tgt_seq_len,
        device,
        lambda msg: batch_iterator.write(msg),
        global_step,
        writer,
    )

    # Save the model at the end of every epoch
    model_filename = trn.get_weights_file_path(CONFIG, f"{epoch:02d}")
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "global_step": global_step,
        },
        model_filename,
    )

No model to preload, starting from scratch


Processing Epoch 00:   0%|          | 6/10515 [00:04<2:20:01,  1.25it/s, loss=9.444] 


--------------------------------------------------------------------------------
    SOURCE: "Well, yes, Planchet, to be sure," said Athos, "what is there so astonishing in that?
    TARGET: Pues claro, Planchet dijo Athos . ¿Qué hay de sorprendente en ello?
 PREDICTED: , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , ,
